# Multi-Omic Fusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/03_multi_omic_fusion.ipynb)

**What this does:** Combines pathway scores from multiple data types (e.g., VCF variant burden + RNA-seq expression) into a unified feature matrix for joint subtype discovery.

**Three fusion strategies:**
- **Concatenate** — column-bind all features with modality prefixes (preserves all info)
- **Weighted Average** — weighted mean of shared pathways across modalities
- **Intersection Only** — restrict to shared samples and pathways

**Prerequisites:** [00_quick_demo.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/00_quick_demo.ipynb), [02_expression_scoring.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/02_expression_scoring.ipynb)

In [ ]:
# Install pathway-subtyping
!pip install -q pathway-subtyping==0.3.1

import pathway_subtyping
print(f"pathway-subtyping v{pathway_subtyping.__version__}")

## 1. Generate Multi-Modal Data

We'll simulate two modalities for the same cohort: variant burden scores (VCF-based) and expression scores (RNA-seq-based). Both share the same samples and pathway definitions.

In [ ]:
from pathway_subtyping import (
    SimulationConfig,
    generate_synthetic_data,
    ExpressionSimulationConfig,
    generate_synthetic_expression_data,
    ExpressionInputType,
    score_pathways_from_expression,
    ExpressionScoringMethod,
)
import numpy as np

# Modality 1: VCF-based pathway burden scores
vcf_sim = generate_synthetic_data(SimulationConfig(
    n_samples=120,
    n_pathways=8,
    n_genes_per_pathway=20,
    n_subtypes=3,
    effect_size=1.2,
    noise_level=1.0,
    seed=42,
))

# Modality 2: Expression-based pathway scores
expr_sim = generate_synthetic_expression_data(ExpressionSimulationConfig(
    n_samples=120,
    n_genes=400,
    n_pathways=8,
    n_genes_per_pathway=30,
    n_subtypes=3,
    effect_size=1.5,
    noise_level=1.0,
    seed=42,
    input_type=ExpressionInputType.TPM,
))

# Score expression
expr_scores = score_pathways_from_expression(
    expr_sim.expression_matrix,
    expr_sim.pathways,
    method=ExpressionScoringMethod.SSGSEA,
    seed=42,
).pathway_scores

# Align column names so pathways match between modalities
vcf_scores = vcf_sim.pathway_scores
vcf_scores.columns = expr_scores.columns[:len(vcf_scores.columns)]

print(f"VCF scores:        {vcf_scores.shape}")
print(f"Expression scores: {expr_scores.shape}")
print(f"Shared pathways:   {len(set(vcf_scores.columns) & set(expr_scores.columns))}")
print(f"Shared samples:    {len(set(vcf_scores.index) & set(expr_scores.index))}")

## 2. Prepare and Fuse Modalities

Each data source is wrapped as a `PreparedModality`, then fused into a single feature matrix.

In [ ]:
from pathway_subtyping import (
    ModalityType,
    FusionStrategy,
    prepare_modality,
    fuse_modalities,
)

# Wrap each data source
vcf_mod = prepare_modality(ModalityType.VCF, vcf_scores, label="WES")
expr_mod = prepare_modality(ModalityType.EXPRESSION, expr_scores, label="RNAseq")

print(f"VCF modality:  {vcf_mod.label}, {vcf_mod.pathway_scores.shape}")
print(f"Expr modality: {expr_mod.label}, {expr_mod.pathway_scores.shape}")

# Strategy 1: Concatenate (recommended — preserves all information)
concat_result = fuse_modalities(
    [vcf_mod, expr_mod],
    strategy=FusionStrategy.CONCATENATE,
    seed=42,
)

print(f"\nCONCATENATE fusion:")
print(f"  Fused shape:    {concat_result.fused_pathway_scores.shape}")
print(f"  Columns:        {list(concat_result.fused_pathway_scores.columns[:4])} ...")
print(f"  Strategy:       {concat_result.strategy.value}")
print(f"  Modalities:     {concat_result.modality_labels}")
print(f"  Quality score:  {concat_result.quality_metrics.get('sample_coverage', 'N/A')}")

In [ ]:
# Strategy 2: Weighted Average (shared pathways only)
weighted_result = fuse_modalities(
    [vcf_mod, expr_mod],
    strategy=FusionStrategy.WEIGHTED_AVERAGE,
    seed=42,
)

# Strategy 3: Intersection Only (strictest — shared samples AND pathways)
intersect_result = fuse_modalities(
    [vcf_mod, expr_mod],
    strategy=FusionStrategy.INTERSECTION_ONLY,
    seed=42,
)

print("Strategy Comparison")
print(f"  {'Strategy':<22} {'Samples':>8} {'Features':>10}")
print(f"  {'-'*42}")
print(f"  {'Concatenate':<22} {concat_result.fused_pathway_scores.shape[0]:>8} {concat_result.fused_pathway_scores.shape[1]:>10}")
print(f"  {'Weighted Average':<22} {weighted_result.fused_pathway_scores.shape[0]:>8} {weighted_result.fused_pathway_scores.shape[1]:>10}")
print(f"  {'Intersection Only':<22} {intersect_result.fused_pathway_scores.shape[0]:>8} {intersect_result.fused_pathway_scores.shape[1]:>10}")

## 3. Cluster Fused Data

We'll cluster the concatenated features and compare subtype recovery across strategies.

In [ ]:
from pathway_subtyping import run_clustering, ClusteringAlgorithm
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

strategies = {
    "Concatenate": concat_result.fused_pathway_scores,
    "Weighted Avg": weighted_result.fused_pathway_scores,
    "Intersection": intersect_result.fused_pathway_scores,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, scores) in zip(axes, strategies.items()):
    cl = run_clustering(scores.values, n_clusters=3, algorithm=ClusteringAlgorithm.GMM, seed=42)
    ari = adjusted_rand_score(vcf_sim.true_labels[:len(cl.labels)], cl.labels)

    pca = PCA(n_components=2, random_state=42)
    X = pca.fit_transform(scores.values)

    for label in sorted(set(cl.labels)):
        mask = cl.labels == label
        ax.scatter(X[mask, 0], X[mask, 1], label=f"Cluster {label}", s=30, alpha=0.7)

    ax.set_title(f"{name}\nARI={ari:.3f}, Sil={cl.silhouette:.3f}")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
    ax.legend(fontsize=8)

plt.suptitle("Fusion Strategies Compared", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Cross-Modal Validation (Gate 5)

Gate 5 checks whether the same subtypes emerge from each modality independently. High cross-modal concordance means the subtypes are biologically robust — not artifacts of a single data type.

In [ ]:
from pathway_subtyping import ValidationGates

gates = ValidationGates(seed=42, n_permutations=50, n_bootstrap=50)

# Use the concatenated result — it preserves per-modality scores
val = gates.run_all(
    pathway_scores=concat_result.fused_pathway_scores,
    cluster_labels=run_clustering(
        concat_result.fused_pathway_scores.values, n_clusters=3, seed=42
    ).labels,
    pathways=vcf_sim.pathways,
    gene_burdens=vcf_sim.gene_burdens,
    n_clusters=3,
    gmm_seed=42,
    per_modality_scores=concat_result.per_modality_scores,
)

print("Validation Gates (including Cross-Modal Gate 5)")
print("=" * 55)
for test in val.results:
    icon = "PASS" if test.passed else "FAIL"
    print(f"  [{icon}] {test.name}: {test.metric_name}={test.metric_value:.3f}")
print(f"\n  Overall: {'ALL PASSED' if val.all_passed else 'SOME FAILED'}")

## 5. Using Your Own Data

```python
from pathway_subtyping import (
    load_expression_matrix, score_pathways_from_expression,
    ExpressionScoringMethod, ExpressionInputType,
    ModalityType, FusionStrategy, prepare_modality, fuse_modalities,
)

# Load expression data
expr, _ = load_expression_matrix("expression.csv", input_type=ExpressionInputType.TPM)
expr_scores = score_pathways_from_expression(expr, pathways, method=ExpressionScoringMethod.SSGSEA, seed=42)

# Load VCF-based scores (from pipeline or manual)
vcf_scores = pd.read_csv("vcf_pathway_scores.csv", index_col=0)

# Fuse
vcf_mod = prepare_modality(ModalityType.VCF, vcf_scores, label="WES")
expr_mod = prepare_modality(ModalityType.EXPRESSION, expr_scores.pathway_scores, label="RNA-seq")
fused = fuse_modalities([vcf_mod, expr_mod], strategy=FusionStrategy.CONCATENATE, seed=42)

# Cluster the fused result
clustering = run_clustering(fused.fused_pathway_scores.values, n_clusters=3, seed=42)
```

## Next Steps

- **Deconvolution:** [04_deconvolution.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/04_deconvolution.ipynb) — add cell-type awareness
- **Visualization:** [05_visualization.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/05_visualization.ipynb) — interactive reports
- **API reference:** [Multi-Omic API](https://codeberg.org/pathways/pathway-subtyping-framework/blob/main/docs/api/multi_omic.md) | [Cross-Modal Validation API](https://codeberg.org/pathways/pathway-subtyping-framework/blob/main/docs/api/cross_modal_validation.md)

---
*Built with [pathway-subtyping](https://pypi.org/project/pathway-subtyping/). Disease-agnostic. Open source.*